# Ocena zapisanych checkpointów segmentacji

Oceniamy checkpoint na stałym podziale walidacyjnym LGG MRI, na CPU.
Model, preprocessing, progi percentylowe i próg predykcji odtwarzamy
z metadanych. Obliczenia wykonują wspólne funkcje z src; notebook
nie utrzymuje osobnych implementacji datasetu, normalizacji ani metryk.

Obsługiwane są dzielenie przez 255, z-score kanałami pacjenta oraz
percentyle + z-score. Dla starszych checkpointów bez config odczytujemy
pole preprocessing, domyślnie scale_255; architekturę historycznego
U-Neta lub MiniUNeta rozpoznajemy po kluczach wag.

Wskaż checkpoint_path i wykonaj komórki od góry w kernelu Python
(inzynierka). Eksport jest domyślnie wyłączony.
Test zostanie oceniony po wspólnym zamknięciu eksperymentów
i zamrożeniu protokołu.

## 1. Biblioteki i podział walidacyjny

Łączymy metadane z istniejącym splits.csv. Wszystkie przekroje jednego
pacjenta pozostają w przypisanej części. Notebook wybiera wyłącznie val.

In [ ]:
from pathlib import Path
import sys

import pandas as pd
import torch

project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.data import build_patient_params
from src.models import build_model
from src.evaluation import evaluate_patients_3d, summarize_evaluation

device = torch.device("cpu")

In [ ]:
pairs_df = pd.read_csv(project_root / "data" / "metadata" / "pairs.csv")
splits_df = pd.read_csv(project_root / "data" / "splits" / "splits.csv")
pairs_with_split = pairs_df.merge(
    splits_df, on="patient_id", how="left", validate="many_to_one"
)
if pairs_with_split["split"].isna().any():
    raise ValueError("Część przekrojów nie ma przydziału w splits.csv.")
val_df = pairs_with_split.loc[pairs_with_split["split"] == "val"].copy()

## 2. Checkpoint i jego konfiguracja

Nowe checkpointy przechowują architekturę, preprocessing, percentyle,
batch size i próg w config. Starsze formaty używają wartości
historycznych: batch size 4 i próg > 0.5. Nieznana architektura lub
metoda preprocessingu zgłasza błąd zamiast uruchamiać inną normalizację.

In [ ]:
checkpoint_path = (
    project_root / "outputs" / "checkpoints"
    / "unet_sampling_percentile_zscore_dice3d_seed42_best.pt"
)
export_results = False
overwrite_exports = False

In [ ]:
checkpoint = torch.load(
    checkpoint_path, map_location="cpu", weights_only=True
)
checkpoint_config = checkpoint.get("config") or {}
state = checkpoint["model_state_dict"]
model_name = checkpoint_config.get("model")
if model_name is None:
    if "encoder1.layers.0.weight" in state:
        model_name = "unet"
    elif "encoder.layers.0.weight" in state:
        model_name = "miniunet"
    else:
        raise ValueError("Brak nazwy modelu i nierozpoznana architektura wag.")

preprocessing = checkpoint_config.get(
    "preprocessing", checkpoint.get("preprocessing", "scale_255")
)
percentiles = checkpoint_config.get("percentiles")
threshold = float(checkpoint_config.get("threshold", 0.5))
batch_size = int(checkpoint_config.get("batch_size", 4))

model = build_model(model_name).to(device)
model.load_state_dict(state, strict=True)
model.eval()

print(f"Checkpoint: {checkpoint_path.name}")
print(f"Model: {model_name}, seed: {checkpoint['seed']}, epoka: {checkpoint['epoch']}")
print(f"Preprocessing: {preprocessing}, percentyle: {percentiles}")
print(f"Urządzenie: {device}, batch size: {batch_size}, próg: > {threshold}")
print("Kryterium wyboru:", checkpoint_config.get(
    "selection_metric", "niezapisane; sprawdź protokół historycznej serii"
))

Checkpoint: unet_sampling_percentile_zscore_dice3d_seed42_best.pt
Model: unet, seed: 42, epoka: 33
Preprocessing: patient_percentile_zscore, percentyle: [10, 99]
Urządzenie: cpu, batch size: 4, próg: > 0.5
Kryterium wyboru: dice_3d


## 3. Parametry obrazów każdego pacjenta

Każdy pacjent otrzymuje parametry wyznaczone wyłącznie ze swoich
obrazów, bez używania masek. Przy percentylach wspólne progi 10/99
obejmują cały przekazany wolumen, kanały i tło. Po przeskalowaniu
i przycięciu do [0, 1] liczymy średnie oraz odchylenia kanałów
w float64. Do modelu trafiają obrazy float32.

In [ ]:
match preprocessing:
    case "scale_255":
        patient_params = None
    case "patient_channel_zscore":
        patient_params = build_patient_params(val_df, project_root)
    case "patient_percentile_zscore":
        if percentiles is None or len(percentiles) != 2:
            raise ValueError("Checkpoint percentylowy wymaga dwóch poziomów percentyli.")
        patient_params = build_patient_params(
            val_df, project_root, percentiles=percentiles
        )
    case _:
        raise ValueError(f"Nieznany preprocessing checkpointu: {preprocessing}")

print("Pacjenci z parametrami:", 0 if patient_params is None else len(patient_params))

Pacjenci z parametrami: 16


## 4. Ocena i pięć metryk

Wspólny moduł sortuje przekroje po slice_id i zbiera wolumeny predykcji.
Trzy wyniki Dice są średnimi po przekrojach: wszystkich, oznaczonych
i pustych wzorcach. Dwie puste maski dają 1, a jedna pusta daje 0.
Dice 3D liczymy z całego wolumenu i uśredniamy z jednakową wagą
każdego pacjenta. Postprocessing zachowuje największą składową 3D
przy sąsiedztwie 6.

Próg i preprocessing pochodzą z checkpointu. Przy porównaniu z historią
treningową na CUDA możliwe są drobne różnice względem raportu CPU.

In [ ]:
patient_results, all_dice, all_annotations = evaluate_patients_3d(
    model,
    val_df,
    project_root,
    device,
    patient_params=patient_params,
    batch_size=batch_size,
    threshold=threshold,
)
summary = summarize_evaluation(patient_results, all_dice, all_annotations)
print(
    f"Pacjenci: {len(patient_results)}, przekroje: {all_dice.numel()}, "
    f"z adnotacją: {all_annotations.sum().item()}"
)
pd.Series(summary, name="Walidacja").to_frame().round(6)

Pacjenci: 16, przekroje: 615, z adnotacją: 236


,Walidacja
dice,0.681487
dice_annotated,0.733536
dice_empty,0.649077
dice_3d,0.708114
dice_3d_filtered,0.752500


## 5. Wyniki poszczególnych pacjentów

Tabela pozwala ocenić, dla których pacjentów zachowanie największej
składowej pomaga lub pogarsza wynik.

In [ ]:
patient_results.round(6)

,patient_id,dice_3d,dice_3d_filtered
0,TCGA_CS_5396_20010302,0.317117,0.322726
1,TCGA_CS_6665_20010817,0.439975,0.472163
2,TCGA_DU_5849_19950405,0.851955,0.933534
3,TCGA_DU_6407_19860514,0.843689,0.927050
4,TCGA_DU_7013_19860523,0.451298,0.445415
5,TCGA_DU_7304_19930325,0.768138,0.778946
6,TCGA_DU_8164_19970111,0.791891,0.836388
7,TCGA_DU_A5TP_19970614,0.784409,0.850069
8,TCGA_DU_A5TU_19980312,0.749781,0.770198
9,TCGA_FG_6691_20020405,0.776354,0.925913


## 6. Opcjonalny eksport

Włącz export_results, aby zapisać raporty do outputs.
Nazwa pliku pochodzi z checkpoint_path.stem: usuwamy końcowe _best,
a _last zachowujemy. Nowe checkpointy z config eksportują jeden wiersz
z run_name, seed, epoch i pięcioma metrykami, jak run_experiments.
Dla starszych checkpointów zachowujemy pionowy format z polskimi
etykietami i kolumną Walidacja, używany w notebookach 09–10.

Istniejące raporty są chronione przez overwrite_exports=False.
Celowe zastąpienie raportów wymaga ustawienia tej wartości na True.

In [ ]:
result_name = checkpoint_path.stem.removesuffix("_best")
output_dir = project_root / "outputs"
metrics_path = output_dir / f"{result_name}_val_metrics.csv"
patients_path = output_dir / f"{result_name}_val_patients.csv"

if export_results:
    if not overwrite_exports:
        for path in (metrics_path, patients_path):
            if path.exists():
                raise FileExistsError(f"Raport już istnieje: {path}")
    output_dir.mkdir(parents=True, exist_ok=True)
    if checkpoint_config:
        run_result = {
            "run_name": checkpoint.get("run_name", result_name),
            "seed": checkpoint["seed"],
            "epoch": checkpoint["epoch"],
            **summary,
        }
        pd.DataFrame([run_result]).to_csv(metrics_path, index=False)
    else:
        legacy_labels = {
            "Dice po wszystkich przekrojach": "dice",
            "Dice przekrojów z adnotacją": "dice_annotated",
            "Dice przekrojów z pustą maską": "dice_empty",
            "Dice 3D po pacjentach": "dice_3d",
            "Dice 3D po pacjentach po filtrowaniu": "dice_3d_filtered",
        }
        pd.Series(
            {label: summary[key] for label, key in legacy_labels.items()},
            name="Walidacja",
        ).to_csv(metrics_path)
    patient_results.to_csv(patients_path, index=False)
    print(metrics_path)
    print(patients_path)
else:
    print("Eksport wyłączony. Wyniki znajdują się w summary i patient_results.")

Eksport wyłączony. Wyniki znajdują się w summary i patient_results.
